In the previous post, we deconstructed a [payment transaction](deconstructing-a-payment-transaction/). Lets build on that, and look at verifying a transaction signature. 

The transaction has 11 inputs, and below are the details for the first input we are going to analyze.
```text 
Input 1 of 11 (bytes 5 to 184)
  Previous TXID : 3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404
  Output index  : 0  (bytes: 00 00 00 00)
  Script length : 139 bytes  (encoded as: 8b)
  Script bytes  :
    48 30 45 02 21 00 c6 d5 92 90 df 93 4b ac 08 bc
    41 24 b0 61 54 ee 21 39 4d 37 ec 95 8d 14 51 29
    a8 85 db d6 3e ab 02 20 03 62 51 12 0e 4c ea cc
    8e 9f f1 35 85 04 99 45 c7 db db 85 92 1a 6e 07
    7a 80 c7 e4 10 93 1f 4a 01 41 04 f3 ce 89 48 c5
    92 69 0c b3 6a 7e 6e 31 53 49 27 56 9f 41 e4 5c
    77 fb db 95 fd b0 e2 35 71 7a 29 5b 2a 28 1c 04
    61 91 39 c4 53 dd 91 39 e9 51 08 63 25 b4 02 1b
    d2 8d a6 fc 3d a8 69 43 cd 04 cf
  Sequence      : 4294967295  (bytes: ff ff ff ff)
```

There is no simple query to find which block a transaction exists on in the blockchain! There is no direct lookup that a node can perform. So, Bitcoin Core creates two databases, a UTXO DB (outputs that have not been spent), and an optional transaction index DB. Because I'm running a pruned node, I don't have the transaction index DB. So, i've used a block explorer to find the block hash for the above transaction.

Now that we have the block hash and the transaction hash, we can use the `getrawtransaction` command to get the data from the block.

In [8]:
# $ bitcoin-cli getblockfrompeer 00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0 0
# $ bitcoin-cli getrawtransaction 3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404 true 00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0

In [15]:
previous_tx = {
  "in_active_chain": True,
  "txid": "3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404",
  "hash": "3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404",
  "version": 1,
  "size": 225,
  "vsize": 225,
  "weight": 900,
  "locktime": 0,
  "vin": [
    {
      "txid": "4e598fea2110bc7f7b1c433db4f9a489ff1a2ba68874000fe9ac8da186c318b7",
      "vout": 1,
      "scriptSig": {
        "asm": "3046022100dbff78061770bf76984c4ea7de28124b3443b96247448d9138710bbb81572cef022100af268d05fbffa315c51811c4c6baf747b6083d62eaaf12762530997c7250700b[ALL] 04337d915c620beac43c8d358158f777ea4d040099e3deb67b63a8e61361ff4f43847b05b5ee971fa3aeb748d3e553dce24b0978831984d9a0b1dbdb648438c34c",
        "hex": "493046022100dbff78061770bf76984c4ea7de28124b3443b96247448d9138710bbb81572cef022100af268d05fbffa315c51811c4c6baf747b6083d62eaaf12762530997c7250700b014104337d915c620beac43c8d358158f777ea4d040099e3deb67b63a8e61361ff4f43847b05b5ee971fa3aeb748d3e553dce24b0978831984d9a0b1dbdb648438c34c"
      },
      "sequence": 4294967295
    }
  ],
  "vout": [
    {
      "value": 50000.00000000,
      "n": 0,
      "scriptPubKey": {
        "asm": "OP_DUP OP_HASH160 fef5cca40eaf93988c09ee70728e139f65f009ff OP_EQUALVERIFY OP_CHECKSIG",
        "desc": "addr(1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB)#7tjw2xwp",
        "hex": "76a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac",
        "address": "1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB",
        "type": "pubkeyhash"
      }
    }
  ],
  "hex": "0100000001b718c386a18dace90f007488a62b1aff89a4f9b43d431c7b7fbc1021ea8f594e010000008c493046022100dbff78061770bf76984c4ea7de28124b3443b96247448d9138710bbb81572cef022100af268d05fbffa315c51811c4c6baf747b6083d62eaaf12762530997c7250700b014104337d915c620beac43c8d358158f777ea4d040099e3deb67b63a8e61361ff4f43847b05b5ee971fa3aeb748d3e553dce24b0978831984d9a0b1dbdb648438c34cffffffff01005039278c0400001976a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac00000000",
  "blockhash": "00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0",
  "confirmations": 720551,
  "time": 1315755730,
  "blocktime": 1315755730
}

previous_tx_output = previous_tx["vout"][0]
previous_tx_output

{'value': 50000.0,
 'n': 0,
 'scriptPubKey': {'asm': 'OP_DUP OP_HASH160 fef5cca40eaf93988c09ee70728e139f65f009ff OP_EQUALVERIFY OP_CHECKSIG',
  'desc': 'addr(1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB)#7tjw2xwp',
  'hex': '76a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac',
  'address': '1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB',
  'type': 'pubkeyhash'}}

## What I learnt

- The blockchain is a database, but it's not like a normal DB. Instead, it's more like a snapshot to verify that things existed in a state on a previous date. For example, there is no simple query to find which block a transaction exists on! There is no direct lookup that a node can perform. So, Bitcoin Core creates two databases, a UTXO DB (outputs that have not been spent), and an optional transaction index DB. Because I'm running a pruned node, I don't have the transaction index DB.
- Because of the pruned node, I have to request any transaction, blocks, etc. from peers who are running a full archival node. But I still can't request a transaction from a peer without a block hash.
- Everything in the blockchain is stored in bytes. The bitcoin-cli and API convert it into hex text while displaying it for me.

  <div class="byte-example">
  <div class="byte-example-row">
  <code>00000000000005c6aca89f188819553e63fe1d5deeea9722b43d1cd0974c7d58</code>
  <span class="byte-example-label">Block hash in hex</span>
  </div>
  <div class="byte-example-row">
  <code>01 00 00 00 f6 05 25 43 53 a7 6d 40 6d 62 8a 86 45 f7 5a 4b f6 48 64 61 c8 1e 76 b7 60 02 00 00 00 00 00 00 4f c5 cd ab 3d 9d 02 e4 bc 62 0a b7 97 80 9c 88 95 d7 6e 5b 9a 54 8c a9 8d f2 86 00 ec 8c 5f 98 23 d5 6c 4e e3 6f 09 1a 23 d7 3f e4</code>
  <span class="byte-example-label">Block header in hex</span>
  </div>
  </div>
- In a transaction, an input spends a previous output. That previous output held the funds, and it has some spending conditions, which the spender has to pass to spend its funds. The input doesn't contain the amount it spent. It only has a reference to the previous output, which has the amount it spent.
- When I downloaded Bitcoin Core, it also provided me with an API called JSON-RPC. Basically, I can send a JSON request to my local Bitcoin Core, and it runs it and will return a response.
- Each block has a block header (80 bytes), which consists of the version (4 bytes), previous block header hash (32 bytes), merkle root hash (32 bytes), time (4 bytes), nBits (4 bytes), and nonce (4 bytes).

  <div class="byte-example">
  <div class="byte-example-row">
  <code>01 00 00 00</code>
  <span class="byte-example-label">version (4 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>f6 05 25 43 53 a7 6d 40 6d 62 8a 86 45 f7 5a 4b f6 48 64 61 c8 1e 76 b7 60 02 00 00 00 00 00 00</code>
  <span class="byte-example-label">previous block header hash (32 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>4f c5 cd ab 3d 9d 02 e4 bc 62 0a b7 97 80 9c 88 95 d7 6e 5b 9a 54 8c a9 8d f2 86 00 ec 8c 5f 98</code>
  <span class="byte-example-label">merkle root hash (32 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>23 d5 6c 4e</code>
  <span class="byte-example-label">time (4 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>e3 6f 09 1a</code>
  <span class="byte-example-label">nBits (4 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>23 d7 3f e4</code>
  <span class="byte-example-label">nonce (4 bytes)</span>
  </div>
  </div>
- After the 80-byte block header, the next bytes represent the transaction count, which tells us how many transactions are included in the block. But the transaction count uses CompactSize which is not fixed. Basically, if the first byte falls between 00-'fc', then it means read only the 81st byte. If the first byte is 'fd', then read the next 2 bytes (82nd and 83rd), for 'fe' read next 4 bytes, for 'ff' read next 8 bytes. Or if it's easier to think in integers.. 0-252 is first.. then 253, 254 and 255 are the markers.
- I’m not using decoderawtransaction from Bitcoin Core for this problem.. So, looking up a previous transaction in an older block isn't very straightforward. For eg, each outpoint in a transaction input has the previous transaction id. BUT, this transaction id is actually a hash of all the data of that previous transaction. It doesn’t exist inside the previous transaction. It's there in the blocks merkle root which allows us to verify that the transaction exists in the block. But as I want to find selected transactions to read the outputs from it, I have to hash each one of the blocks transactions, create the transaction hash first, and if chosen, get the transaction outputs. For each transaction I want, the block already has the exact bytes I need to hash. But I need to find the start and position for each transaction from the block.
- To find the start, end for each transaction.. I need to increment the position by the correct field size. Some of the transaction fields are or fixed bytes.. e.g. version, lock\_time, the previous\_output etc.. But many of them are of variable size of type compactSize. For e.g. the number of transactions in a block, the number of input transactions in a transaction, the public\_key script. So, the program needs to update the position by finding the compactSize for each of them. Doing all of this will give us the start and end position for each transaction.. Which we can then use to find the exact bytes for each transaction and then hash it to get the id.

